In [1]:
# 01_data_loading.ipynb
# Cross-Domain AI Governance Gap Analysis
# 01. Data Loading
#
# Load four datasets for the credit vs insurance cross-domain comparison.
#   [credit]    German Credit (UCI) ............ sensitive: sex, age
#   [credit]    Credit Card Approval (Kaggle) ... sensitive: sex; targets from credit_record
#   [insurance] Medical Cost Personal (Kaggle) .. sensitive: sex, age, smoker
#   [insurance] Porto Seguro Safe Driver (Kaggle) indirect proxy variables (ps_ind_*)

import os
import urllib.request
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# All paths are relative to the notebooks/ folder (never printed, for anonymity)
DATA_DIR = os.path.join("..", "data")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Download German Credit from UCI only if absent. The Kaggle files must be placed
# in data/ manually: application_record.csv, credit_record.csv, insurance.csv, porto_train.csv
german_file = os.path.join(DATA_DIR, "german.data")
if not os.path.exists(german_file):
    urllib.request.urlretrieve(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data",
        german_file,
    )

# German Credit (Statlog), UCI [credit domain]
german_cols = [
    "status_checking", "duration", "credit_history", "purpose", "credit_amount",
    "savings", "employment", "installment_rate", "personal_status_sex", "other_debtors",
    "residence_since", "property", "age", "other_installment", "housing",
    "existing_credits", "job", "num_dependents", "telephone", "foreign_worker", "target",
]
df_german = pd.read_csv(german_file, sep=r"\s+", header=None, names=german_cols)
# Recode target: original 1 = good, 2 = bad -> 1 = bad (default), 0 = good
df_german["target"] = (df_german["target"] == 2).astype(int)
sex_map = {"A91": "male", "A92": "female", "A93": "male", "A94": "male", "A95": "female"}
df_german["sex"] = df_german["personal_status_sex"].map(sex_map)
df_german["age_group"] = np.where(df_german["age"] >= 25, "old", "young")
df_german["domain"] = "credit"
print("German Credit shape:", df_german.shape)

# Credit Card Approval Prediction, Kaggle [credit domain]
# STATUS codes: 0=1-29d,1=30-59d,2=60-89d,3=90-119d,4=120-149d,5=150+/writeoff,C=paid,X=no loan
# target_30dpd : bad if any month reaches 30+ dpd (STATUS in {1,2,3,4,5})
# target_60dpd : bad if any month reaches 60+ dpd (STATUS in {2,3,4,5})
PRIMARY_TARGET = "target_30dpd"
df_app = pd.read_csv(os.path.join(DATA_DIR, "application_record.csv"))
df_credit = pd.read_csv(os.path.join(DATA_DIR, "credit_record.csv"))
status_num = df_credit["STATUS"].replace({"C": -1, "X": -1}).astype(int)
df_credit = df_credit.assign(status_num=status_num)
worst = df_credit.groupby("ID")["status_num"].max()
labels = pd.DataFrame({
    "target_30dpd": (worst >= 1).astype(int),
    "target_60dpd": (worst >= 2).astype(int),
})
df_home = df_app.merge(labels, on="ID", how="inner")
df_home["target"] = df_home[PRIMARY_TARGET]
df_home["sex"] = df_home["CODE_GENDER"].map({"M": "male", "F": "female"})
df_home = df_home[df_home["sex"].notna()].copy()
df_home["domain"] = "credit"
print("Credit Card Approval shape:", df_home.shape, "| primary target:", PRIMARY_TARGET)

# Medical Cost Personal, Kaggle [insurance domain]
# Continuous charges -> binary underwriting-style target (top 25% = high risk)
df_ins = pd.read_csv(os.path.join(DATA_DIR, "insurance.csv"))
df_ins["sex"] = df_ins["sex"].str.lower()
df_ins["age_group"] = np.where(df_ins["age"] >= 40, "old", "young")
df_ins["smoker_flag"] = (df_ins["smoker"] == "yes").astype(int)
threshold = df_ins["charges"].quantile(0.75)
df_ins["target"] = (df_ins["charges"] >= threshold).astype(int)
df_ins["domain"] = "insurance"
print("Medical Cost shape:", df_ins.shape)

# Porto Seguro Safe Driver, Kaggle [insurance domain]
# No explicit sensitive attribute; anonymized individual proxy variables (ps_ind_* cat/bin)
df_porto = pd.read_csv(os.path.join(DATA_DIR, "porto_train.csv"))
df_porto = df_porto.replace(-1, np.nan)           # -1 is the missing indicator
df_porto["domain"] = "insurance"
proxy_cols = [c for c in df_porto.columns if c.startswith("ps_ind") and ("cat" in c or "bin" in c)]
print("Porto Seguro shape:", df_porto.shape, "| n proxy candidates:", len(proxy_cols))

# Cross-domain summary table
summary = pd.DataFrame([
    {"domain": "credit", "dataset": "German Credit", "n_rows": len(df_german),
     "pos_rate": round(df_german["target"].mean(), 3), "sensitive": "sex, age_group"},
    {"domain": "credit", "dataset": "Credit Card Approval (30dpd)", "n_rows": len(df_home),
     "pos_rate": round(df_home["target_30dpd"].mean(), 3), "sensitive": "sex"},
    {"domain": "credit", "dataset": "Credit Card Approval (60dpd)", "n_rows": len(df_home),
     "pos_rate": round(df_home["target_60dpd"].mean(), 3), "sensitive": "sex"},
    {"domain": "insurance", "dataset": "Medical Cost", "n_rows": len(df_ins),
     "pos_rate": round(df_ins["target"].mean(), 3), "sensitive": "sex, age_group, smoker"},
    {"domain": "insurance", "dataset": "Porto Seguro", "n_rows": len(df_porto),
     "pos_rate": round(df_porto["target"].mean(), 3), "sensitive": "proxy (ps_ind_*)"},
])
print(summary.to_string(index=False))
summary.to_csv(os.path.join(TAB_DIR, "01_crossdomain_summary.csv"), index=False)

# Persist intermediate frames for the next notebook
df_german.to_parquet(os.path.join(PROCESSED_DIR, "german.parquet"))
df_home.to_parquet(os.path.join(PROCESSED_DIR, "credit_card_approval.parquet"))
df_ins.to_parquet(os.path.join(PROCESSED_DIR, "insurance.parquet"))
df_porto.to_parquet(os.path.join(PROCESSED_DIR, "porto_seguro.parquet"))
print("01_data_loading complete")


German Credit shape: (1000, 24)


Credit Card Approval shape: (36457, 23) | primary target: target_30dpd
Medical Cost shape: (1338, 11)


Porto Seguro shape: (595212, 60) | n proxy candidates: 14
   domain                      dataset  n_rows  pos_rate              sensitive
   credit                German Credit    1000     0.300         sex, age_group
   credit Credit Card Approval (30dpd)   36457     0.118                    sex
   credit Credit Card Approval (60dpd)   36457     0.017                    sex
insurance                 Medical Cost    1338     0.250 sex, age_group, smoker
insurance                 Porto Seguro  595212     0.036       proxy (ps_ind_*)


01_data_loading complete
